In [4]:
!pip -q install pyspark

In [5]:
from google.colab import files
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, when, round

uploaded = files.upload()

spark = SparkSession.builder \
    .appName("LazyExecution") \
    .getOrCreate()

sales_df = spark.read.csv(
    "lazy_execution_sales_25000.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "lazy_execution_products.csv",
    header=True,
    inferSchema=True
)

stores_df = spark.read.csv(
    "lazy_execution_stores.csv",
    header=True,
    inferSchema=True
)

print("Sales records:", sales_df.count())
print("Product records:", products_df.count())
print("Store records:", stores_df.count())

sales_df.printSchema()
products_df.printSchema()
stores_df.printSchema()

Saving lazy_execution_stores.csv to lazy_execution_stores.csv
Saving lazy_execution_sales_25000.csv to lazy_execution_sales_25000 (1).csv
Saving lazy_execution_products.csv to lazy_execution_products.csv
Sales records: 25000
Product records: 15
Store records: 8
root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_pct: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- sales_channel: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- list_price: integer (nullable = true)
 |-- cost_price: integer (nullable = true)

root
 |-- store_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- re

**Part A**

In [6]:
filtered_df = sales_df.filter(col("quantity") > 5)

print("Transformation created")

Transformation created


In [7]:
selected_df = filtered_df.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
)

selected_df.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
| O000002|      P014|       6|     29999|
| O000003|      P008|       6|     22999|
| O000004|      P008|       7|     22999|
| O000006|      P005|       8|      2499|
| O000007|      P010|       8|      3999|
| O000008|      P002|       9|      2999|
| O000012|      P010|       9|      3999|
| O000013|      P013|       6|      5999|
| O000014|      P005|       8|      2499|
| O000015|      P007|       7|     79999|
| O000017|      P004|       9|     16999|
| O000018|      P012|       7|      6999|
| O000020|      P010|       7|      3999|
| O000021|      P006|       7|     55999|
| O000022|      P003|       7|      1899|
| O000023|      P010|       9|      3999|
| O000024|      P009|       7|      1999|
| O000028|      P014|       7|     29999|
| O000033|      P002|       6|      2999|
| O000034|      P012|       6|      6999|
+--------+----------+--------+----

In [8]:
expensive_df = sales_df.filter(
    col("unit_price") > 20000
)

print("Matching records:", expensive_df.count())

Matching records: 6560


In [9]:
gross_df = sales_df.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)

gross_df.explain()

gross_df.show(10)

== Physical Plan ==
*(1) Project [order_id#44, order_date#45, store_id#46, product_id#47, quantity#48, unit_price#49, discount_pct#50, payment_method#51, order_status#52, sales_channel#53, (quantity#48 * unit_price#49) AS gross_amount#161]
+- FileScan csv [order_id#44,order_date#45,store_id#46,product_id#47,quantity#48,unit_price#49,discount_pct#50,payment_method#51,order_status#52,sales_channel#53] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|gross_amount|
+--------+----------+--------+----------+--------+-

In [10]:
completed_df = sales_df.filter(
    col("order_status") == "Completed"
)

sorted_df = completed_df.orderBy(
    col("unit_price").desc()
)

print(sorted_df.take(10))

[Row(order_id='O000085', order_date=datetime.date(2026, 4, 22), store_id='S03', product_id='P007', quantity=4, unit_price=79999, discount_pct=0, payment_method='Card', order_status='Completed', sales_channel='Online'), Row(order_id='O000290', order_date=datetime.date(2026, 6, 26), store_id='S06', product_id='P007', quantity=3, unit_price=79999, discount_pct=10, payment_method='Net Banking', order_status='Completed', sales_channel='Online'), Row(order_id='O000098', order_date=datetime.date(2026, 4, 15), store_id='S05', product_id='P007', quantity=1, unit_price=79999, discount_pct=5, payment_method='Net Banking', order_status='Completed', sales_channel='Online'), Row(order_id='O000005', order_date=datetime.date(2026, 3, 14), store_id='S06', product_id='P007', quantity=2, unit_price=79999, discount_pct=0, payment_method='Cash', order_status='Completed', sales_channel='Online'), Row(order_id='O000107', order_date=datetime.date(2026, 5, 24), store_id='S05', product_id='P007', quantity=9, un

**Part B**

In [11]:
pipeline_df = sales_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
).select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "gross_amount"
).orderBy(
    col("gross_amount").desc()
)

pipeline_df.show(20)

+--------+----------+--------+----------+------------+
|order_id|product_id|quantity|unit_price|gross_amount|
+--------+----------+--------+----------+------------+
| O002603|      P007|       9|     79999|      719991|
| O006170|      P007|       9|     79999|      719991|
| O002693|      P007|       9|     79999|      719991|
| O000107|      P007|       9|     79999|      719991|
| O002841|      P007|       9|     79999|      719991|
| O000367|      P007|       9|     79999|      719991|
| O002940|      P007|       9|     79999|      719991|
| O000527|      P007|       9|     79999|      719991|
| O003108|      P007|       9|     79999|      719991|
| O000690|      P007|       9|     79999|      719991|
| O003542|      P007|       9|     79999|      719991|
| O001201|      P007|       9|     79999|      719991|
| O003884|      P007|       9|     79999|      719991|
| O001300|      P007|       9|     79999|      719991|
| O003897|      P007|       9|     79999|      719991|
| O001796|

In [13]:
discounted_df = sales_df.filter(
    col("discount_pct") >= 10
).withColumn(
    "discounted_amount",
    col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100)
).orderBy(
    col("discounted_amount").desc()
)

discounted_df.show(15)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|discounted_amount|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+-----------------+
| O006653|2026-06-29|     S01|      P007|       9|     79999|          10|          Card|   Cancelled|       Online|         647991.9|
| O011893|2026-03-26|     S07|      P007|       9|     79999|          10|           UPI|   Completed|       Online|         647991.9|
| O006699|2026-06-25|     S05|      P007|       9|     79999|          10|   Net Banking|   Completed|       Online|         647991.9|
| O003108|2026-02-23|     S08|      P007|       9|     79999|          10|   Net Banking|   Completed|     In Store|         647991.9|
| O007743|2026-05-22|     S08|      P007|       9|     

In [14]:
payment_df = sales_df.groupBy(
    "payment_method"
).agg(
    sum("quantity").alias("total_quantity")
).orderBy(
    col("total_quantity").desc()
)

payment_df.show()

+--------------+--------------+
|payment_method|total_quantity|
+--------------+--------------+
|          Cash|         31754|
|          Card|         31327|
|           UPI|         30894|
|   Net Banking|         30889|
+--------------+--------------+



In [15]:
channel_df = sales_df.groupBy(
    "sales_channel",
    "order_status"
).agg(
    count("*").alias("transaction_count")
)

channel_df.show()

+-------------+------------+-----------------+
|sales_channel|order_status|transaction_count|
+-------------+------------+-----------------+
|       Online|   Completed|            10797|
|     In Store|   Cancelled|              592|
|       Online|    Returned|              987|
|       Online|   Cancelled|              633|
|     In Store|    Returned|             1013|
|     In Store|   Completed|            10978|
+-------------+------------+-----------------+



In [16]:
small_df = sales_df.select(
    "order_id",
    "store_id",
    "product_id",
    "quantity"
)

result_df = small_df.drop("store_id")

print(result_df.first())

Row(order_id='O000001', product_id='P002', quantity=3)


In [17]:
small_df = sales_df.select(
    "order_id",
    "store_id",
    "product_id",
    "quantity"
)

result_df = small_df.drop("store_id")

print(result_df.first())

Row(order_id='O000001', product_id='P002', quantity=3)


**Part C**

In [18]:
sales_products_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
)

sales_products_df.select(
    "product_name",
    "category",
    "quantity",
    "unit_price"
).show(20)

+-------------------+-----------+--------+----------+
|       product_name|   category|quantity|unit_price|
+-------------------+-----------+--------+----------+
|Mechanical Keyboard|Accessories|       3|      2999|
|      Smartphone 5G|    Mobiles|       6|     29999|
|     Android Tablet|    Tablets|       6|     22999|
|     Android Tablet|    Tablets|       7|     22999|
|          Laptop 16|  Computers|       2|     79999|
|          Webcam HD|Accessories|       8|      2499|
|  Bluetooth Speaker|      Audio|       8|      3999|
|Mechanical Keyboard|Accessories|       9|      2999|
|          Laptop 16|  Computers|       5|     79999|
|          Laptop 14|  Computers|       5|     55999|
|          Laptop 16|  Computers|       1|     79999|
|  Bluetooth Speaker|      Audio|       9|      3999|
|   External HDD 2TB|    Storage|       6|      5999|
|          Webcam HD|Accessories|       8|      2499|
|          Laptop 16|  Computers|       7|     79999|
|          Laptop 14|  Compu

In [19]:
sales_stores_df = sales_df.join(
    stores_df,
    on="store_id",
    how="inner"
)

city_sales_df = sales_stores_df.groupBy(
    "city"
).agg(
    sum("quantity").alias("total_quantity")
)

city_sales_df.show()

+---------+--------------+
|     city|total_quantity|
+---------+--------------+
|  Chennai|         15727|
|   Mumbai|         15489|
|  Kolkata|         15619|
|     Pune|         15407|
|    Delhi|         15101|
|Bengaluru|         15718|
|Hyderabad|         15784|
|   Jaipur|         16019|
+---------+--------------+



In [20]:
all_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).join(
    stores_df,
    on="store_id",
    how="inner"
)

category_region_df = all_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "revenue",
    col("quantity") * col("unit_price")
).groupBy(
    "category",
    "region"
).agg(
    sum("revenue").alias("total_revenue")
)

category_region_df.show()

+-----------+------+-------------+
|   category|region|total_revenue|
+-----------+------+-------------+
|   Displays| North|     32468090|
|   Displays| South|     45761308|
|    Storage| South|     36976319|
|   Displays|  East|     13616199|
|      Audio|  East|     12043135|
|    Tablets| North|     38270336|
|    Mobiles| South|    120674524|
|    Storage|  East|     12140136|
|    Mobiles| North|     75482543|
|    Mobiles|  East|     41178126|
|Accessories|  East|      9093361|
|      Audio|  West|     25317111|
|    Tablets|  West|     37488370|
|   Displays|  West|     28286336|
|      Audio| South|     34629665|
|      Audio| North|     24327211|
|    Storage|  West|     23673341|
|  Computers| North|    240756425|
|  Computers|  West|    221092729|
|    Mobiles|  West|     80372388|
+-----------+------+-------------+
only showing top 20 rows


In [24]:
completed_sales_df = sales_df.filter(
    col("order_status") == "Completed"
)

product_revenue_df = completed_sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).withColumn(
    "discounted_revenue",
    col("quantity") * col("unit_price") *
    (1 - col("discount_pct") / 100)
).groupBy(
    "product_id",
    "product_name"
).agg(
    sum("discounted_revenue").alias("total_revenue")
).orderBy(
    col("total_revenue").desc()
)

product_revenue_df.show(5)

+----------+---------------+--------------------+
|product_id|   product_name|       total_revenue|
+----------+---------------+--------------------+
|      P007|      Laptop 16|5.2272546585000014E8|
|      P006|      Laptop 14| 3.830051604999999E8|
|      P014|  Smartphone 5G| 2.060406317500002E8|
|      P008| Android Tablet|1.5040656029999995E8|
|      P004|27-inch Monitor|1.1406074015000013E8|
+----------+---------------+--------------------+
only showing top 5 rows


**Part D**

In [31]:
step1 = sales_df.filter(
    col("quantity") > 5
)
print("Transformation 1 created")

step2 = step1.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price"
)
print("Transformation 2 created")

step3 = step2.withColumn(
    "gross_amount",
    col("quantity") * col("unit_price")
)
print("Transformation 3 created")

step4 = step3.groupBy(
    "product_id"
).agg(
    sum("gross_amount").alias("total_amount")
)
print("Transformation 4 created")

step5 = step4.orderBy(
    col("total_amount").desc()
)
print("Transformation 5 created")

step5.show()


filtered_df = sales_df.filter(
    col("quantity") > 5
)

filtered_df.explain()

filtered_df.show(10)

print("Total matching records:", filtered_df.count())


category_totals_df = sales_df.join(
    products_df,
    on="product_id",
    how="inner"
).groupBy(
    "category"
).agg(
    sum("quantity").alias("total_quantity")
)

category_totals = category_totals_df.collect()

for row in category_totals:
    print(row)


parquet_df = sales_df.filter(
    col("order_status") == "Completed"
).withColumn(
    "discounted_revenue",
    col("quantity") * col("unit_price")
)

parquet_df.write.mode("overwrite").parquet(
    "/content/completed_orders_parquet"
)

print("Parquet file saved successfully")

Transformation 1 created
Transformation 2 created
Transformation 3 created
Transformation 4 created
Transformation 5 created
+----------+------------+
|product_id|total_amount|
+----------+------------+
|      P007|   425994675|
|      P006|   297858681|
|      P014|   167304423|
|      P008|   119410808|
|      P004|    87663843|
|      P015|    76434540|
|      P011|    52239195|
|      P012|    41952006|
|      P013|    34392267|
|      P010|    23414145|
|      P002|    16152614|
|      P005|    14481705|
|      P009|    10802596|
|      P003|    10152054|
|      P001|     4676547|
+----------+------------+

== Physical Plan ==
*(1) Filter (isnotnull(quantity#48) AND (quantity#48 > 5))
+- FileScan csv [order_id#44,order_date#45,store_id#46,product_id#47,quantity#48,unit_price#49,discount_pct#50,payment_method#51,order_status#52,sales_channel#53] Batched: false, DataFilters: [isnotnull(quantity#48), (quantity#48 > 5)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/